<h2><font color="#004D7F" size=5>Módulo 1</font></h2>



<h1><font color="#004D7F" size=6> 3. Stacked Generalization </font></h1>

<br><br>
<div style="text-align: right">
<font color="#004D7F" size=3>Manuel Castillo-Cara</font><br>
<font color="#004D7F" size=3>Aprendizaje Automático II</font><br>
<font color="#004D7F" size=3>Universidad Nacional de Educación a Distancia</font>

</div>

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
More information about [Manuel Castillo-Cara](https://www.manuelcastillo.eu/)

<div class="alert alert-block alert-info">

<i class="fa fa-info-circle" aria-hidden="true"></i>
Puedes ver más cursos de Inteligencia Artificial, Machine Learning y Deep Learning en mi [página web personal](https://www.manuelcastillo.eu/udemy/)

---

<a id="indice"></a>
<h2><font color="#004D7F" size=5>Índice</font></h2>


* [1. Introducción](#section1)
   * [1.1. Algoritmo Stacked Generalization](#section11)
   * [1.2. Dataset](#section12)
   * [1.3. Tutorial](#section13)
* [2. Submodelos y agregador](#section2)
    * [2.1. Submodelo #1: _k_-Nearest Neighbors](#section21)
    * [2.2. Submodelo #2: Perceptrón](#section22)
    * [2.3. Modelo Agregador: Regresión Logística](#section23)
* [3. Combinar predicciones](#section3)
* [4. Caso de estudio: dataset Sonar](#section4)
* [Ejercicios](#sectionEj)

---

<a id="section0"></a>
# <font color="#004D7F">0. Contexto</font>

Los métodos de conjunto son una excelente manera de mejorar el rendimiento predictivo de sus problemas de aprendizaje automático. La generalización apilada o apilamiento es una técnica de conjunto que utiliza un nuevo modelo para aprender cómo combinar mejor las predicciones de dos o más modelos entrenados en su conjunto de datos. 

En este tutorial, descubrirá cómo implementar el apilamiento desde cero en Python. Después de completar este tutorial, sabrá:
- Cómo aprender a combinar las predicciones de múltiples modelos en un conjunto de datos.
- Cómo aplicar la generalización apilada a un problema de modelado predictivo del mundo real.

---
<div style="text-align: right">
<a href="#indice"><font size=5><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></font></a>
</div>

---

<a id="section1"></a>
# <font color="#004D7F"> 1. Introducción</font>

Esta sección proporciona una breve descripción de Bootstrap Aggregation y el conjunto de datos de Sonar que se utilizarán en este tutorial.

<a id="section11"></a>
## <font color="#004D7F"> 1.1. Algoritmo Stacked Generalization</font>

La generalización apilada es un algoritmo de conjunto en el que se entrena un nuevo modelo para combinar las predicciones de dos o más modelos ya entrenados o de su conjunto de datos. 
- Las predicciones de los modelos o submodelos existentes se combinan utilizando un nuevo modelo y, como tal, el apilamiento a menudo se denomina combinación, ya que las predicciones de los submodelos se combinan.
- Es típico utilizar un método lineal simple para combinar las predicciones de los submodelos. Los ejemplos incluyen: 
    - un promedio simple llamado votación, 
    - una suma ponderada mediante regresión lineal y 
    - regresión logística. 
- Los modelos que tienen sus predicciones combinadas deben tener habilidad en el problema, pero no necesitan ser los mejores modelos posibles. 
    - Esto significa que no es necesario ajustar los submodelos intensamente, siempre que el modelo muestre alguna ventaja sobre una predicción de referencia.
- Es importante que los submodelos produzcan predicciones diferentes, las llamadas predicciones no correlacionadas. 
- El apilamiento funciona mejor cuando las predicciones que se combinan son todas hábiles, pero hábiles de diferentes maneras. 
    - Esto se puede lograr mediante el uso de algoritmos que utilizan representaciones internas muy diferentes (árboles en comparación con instancias) y/o modelos entrenados en diferentes representaciones o proyecciones de los datos de entrenamiento.

En este tutorial, veremos cómo tomar dos submodelos muy diferentes y no sintonizados y combinar sus predicciones con un algoritmo de regresión logística simple.

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> __Nota__: En Scikit-learn podemos ver los diferentes parámetros que tiene [StackingClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.StackingClassifier.html) y [StackingRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.StackingRegressor.html)
</div>

<a id="section12"></a>
## <font color="#004D7F"> 1.2. Dataset</font>

En este tutorial usaremos el conjunto de datos Sonar. 
- Este conjunto de datos implica la discriminación entre minas y rocas. 
- El desempeño de referencia sobre el problema es aproximadamente del 53%. 

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> __Nota__: Más información sobre el dataset [Sonar](https://archive.ics.uci.edu/dataset/151/connectionist+bench+sonar+mines+vs+rocks)
</div>

<a id="section13"></a>
## <font color="#004D7F"> 1.3. Tutorial</font>

Este tutorial se divide en 3 partes:
- Submodelos y Agregador.
- Combinar predicciones.
- Caso de estudio Sonar.
Estos pasos proporcionan la base que necesita para implementar y aplicar bootstrap aggregation con árboles de decisión a sus propios problemas de modelado predictivo.

---
<div style="text-align: right">
<a href="#indice"><font size=5><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></font></a>
</div>

---

<a id="section2"></a> 
# <font color="#004D7F"> 2. Submodelos y agregador </font>

Usaremos dos modelos como submodelos para apilar y un modelo lineal como modelo agregador.
Esta parte se divide en 3 secciones:
1. Submodelo #1: _k_-Nearest Neighbors.
2. Submodelo #2: Perceptrón.
3. Modelo Agregador: Regresión Logística.
Cada modelo se describirá en términos de las funciones utilizadas para entrenar el modelo y una función.
Se utiliza para hacer predicciones.

<a id="section21"></a> 
## <font color="#004D7F"> 2.1. Submodelo #1: _k_-Nearest Neighbors </font>

El algoritmo _k_-Nearest Neighbors (KNN) utiliza todo el conjunto de datos de entrenamiento como modelo. Por lo tanto, entrenar el modelo implica conservar el conjunto de datos de entrenamiento. A continuación se muestra una función
llamada `knn_model()` que hace precisamente esto.

In [1]:
# Prepare the kNN model
def knn_model(train):
	return train

Hacer predicciones implica encontrar los _k_ registros más similares en el conjunto de datos de entrenamiento y seleccionar los valores de clase más comunes. La función de distancia euclidiana se utiliza para calcular la similitud entre nuevas filas de datos y las filas del conjunto de datos de entrenamiento.

A continuación se muestran estas funciones auxiliares que implican realizar predicciones para un modelo KNN.
- La función `euclidean_distance()` calcula la distancia entre dos filas de datos, 
- La función `get_neighbors()` localiza todos los vecinos en el conjunto de datos de entrenamiento para una nueva fila de datos y 
- `knn_predict()` hace una predicción de los vecinos para una nueva fila de datos.

In [2]:
# Calculate the Euclidean distance between two vectors
def euclidean_distance(row1, row2):
	distance = 0.0
	for i in range(len(row1)-1):
		distance += (row1[i] - row2[i])**2
	return sqrt(distance)

# Locate neighbors for a new row
def get_neighbors(train, test_row, num_neighbors):
	distances = list()
	for train_row in train:
		dist = euclidean_distance(test_row, train_row)
		distances.append((train_row, dist))
	distances.sort(key=lambda tup: tup[1])
	neighbors = list()
	for i in range(num_neighbors):
		neighbors.append(distances[i][0])
	return neighbors

# Make a prediction with kNN
def knn_predict(model, test_row, num_neighbors=2):
	neighbors = get_neighbors(model, test_row, num_neighbors)
	output_values = [row[-1] for row in neighbors]
	prediction = max(set(output_values), key=output_values.count)
	return prediction

Puede ver que el número de vecinos (_k_) está establecido en 2 como parámetro predeterminado en la función `knn_predict()`. Este número fue elegido con un poco de prueba y error y no fue afinado. Ahora que tenemos los componentes básicos de un modelo KNN, veamos el algoritmo Perceptron.

<a id="section22"></a> 
## <font color="#004D7F"> 2.2. Submodelo #2: Perceptrón </font>

El modelo del algoritmo Perceptron es un conjunto de pesos aprendidos de los datos de entrenamiento. Para entrenar los pesos, es necesario realizar muchas predicciones sobre los datos de entrenamiento para calcular los valores de error. Por lo tanto, tanto el entrenamiento como la predicción del modelo requieren una función de predicción.

A continuación se muestran las funciones auxiliares para implementar el algoritmo Perceptron. 
- La función `perceptron()` entrena el modelo Perceptron en el conjunto de datos de entrenamiento y 
- `perceptron_predict()` se utiliza para hacer una predicción para una fila de datos.

In [3]:
# Make a prediction with weights
def perceptron_predict(model, row):
	activation = model[0]
	for i in range(len(row)-1):
		activation += model[i + 1] * row[i]
	return 1.0 if activation >= 0.0 else 0.0

# Estimate Perceptron weights using stochastic gradient descent
def perceptron_model(train, l_rate=0.01, n_epoch=5000):
	weights = [0.0 for i in range(len(train[0]))]
	for _ in range(n_epoch):
		for row in train:
			prediction = perceptron_predict(weights, row)
			error = row[-1] - prediction
			weights[0] = weights[0] + l_rate * error
			for i in range(len(row)-1):
				weights[i + 1] = weights[i + 1] + l_rate * error * row[i]
	return weights

La función `perceptron_model()` especifica tanto la tasa de aprendizaje como el número de épocas de entrenamiento como parámetros predeterminados. Nuevamente, estos parámetros se eligieron con un poco de prueba y error, pero no se ajustaron al conjunto de datos. 

Ahora tenemos implementaciones para ambos submodelos; veamos la implementación del modelo agregador. Para más información sobre el algoritmo Perceptrón, consulte el Capítulo 10.

<a id="section23"></a> 
## <font color="#004D7F"> 2.3. Modelo Agregador: Regresión Logística</font>

Al igual que el algoritmo Perceptron, la Regresión Logística utiliza un conjunto de ponderaciones, llamados coeficientes, como representación del modelo. Y al igual que el algoritmo Perceptron, los coeficientes se aprenden haciendo predicciones iterativas sobre los datos de entrenamiento y actualizándolos.

A continuación se muestran las funciones auxiliares para implementar el algoritmo de regresión logística. 
- La función `logistic_regression_model()` se usa para entrenar los coeficientes en el conjunto de datos de entrenamiento y
- la función `logistic_regresion_predict()` se usa para hacer una predicción para una fila de datos.

In [4]:
# Make a prediction with coefficients
def logistic_regression_predict(model, row):
	yhat = model[0]
	for i in range(len(row)-1):
		yhat += model[i + 1] * row[i]
	return 1.0 / (1.0 + exp(-yhat))

# Estimate logistic regression coefficients using stochastic gradient descent
def logistic_regression_model(train, l_rate=0.01, n_epoch=5000):
	coef = [0.0 for i in range(len(train[0]))]
	for _ in range(n_epoch):
		for row in train:
			yhat = logistic_regression_predict(coef, row)
			error = row[-1] - yhat
			coef[0] = coef[0] + l_rate * error * yhat * (1.0 - yhat)
			for i in range(len(row)-1):
				coef[i + 1] = coef[i + 1] + l_rate * error * yhat * (1.0 - yhat) * row[i]
	return coef

`logistic_regression_model()` define una tasa de aprendizaje y un número de épocas por defecto parámetros, y al igual que con los otros algoritmos, estos parámetros se encontraron con una pequeña prueba y error y no fueron optimizados. 

Ahora que tenemos implementaciones de submodelos y el modelo agregador, veamos cómo podemos combinar las predicciones de múltiples modelos. Para más sobre la regresión logística, consulte el Capítulo 9.

<a id="section3"></a> 
# <font color="#004D7F"> 3. Combinar predicciones</font>

Para un algoritmo de aprendizaje automático, aprender a combinar predicciones es muy parecido a aprender de un conjunto de datos de entrenamiento. Se puede construir un nuevo conjunto de datos de entrenamiento a partir de las predicciones de los submodelos, de la siguiente manera:
- Cada fila representa una fila en el conjunto de datos de entrenamiento.
- La primera columna contiene predicciones para cada fila del conjunto de datos de entrenamiento realizadas por el primer submodelo, como KNN.
- La segunda columna contiene predicciones para cada fila del conjunto de datos de entrenamiento realizadas por el segundo submodelo, como el algoritmo Perceptron.
- La tercera columna contiene el valor de salida esperado para la fila en el conjunto de datos de entrenamiento.

A continuación se muestra un ejemplo artificial de cómo podría verse un conjunto de datos de apilamiento construido:

| KNN | PER | Y |
|:---:|:---:|:-:|
|  0  |  0  | 0 |
|  1  |  0  | 1 |
|  0  |  1  | 0 |
|  1  |  1  | 1 |
|  0  |  1  | 0 |

Luego se puede entrenar un algoritmo de aprendizaje automático, como la regresión logística, en base a este nuevo conjunto de datos. En esencia, este nuevo metaalgoritmo aprende cómo combinar mejor la predicción de múltiples submodelos. 

A continuación se muestra una función denominada `to_stacked_row()` que implementa esto procedimiento para crear nuevas filas para este conjunto de datos apilados.
- La función toma una lista de modelos como entrada; estos se utilizan para hacer predicciones. 
- La función también toma una lista de funciones como entrada, una función utilizada para hacer una predicción para cada modelo.
- Finalmente, se incluye una sola fila del conjunto de datos de entrenamiento. Se construye una nueva fila en una columna a la vez. 
- Las predicciones se calculan utilizando cada modelo y la fila de datos de entrenamiento. 
- El valor de salida esperado de la fila del conjunto de datos de entrenamiento luego se agrega como la última columna a la fila.

In [5]:
# Make predictions with sub-models and construct a new stacked row
def to_stacked_row(models, predict_list, row):
	stacked_row = list()
	for i in range(len(models)):
		prediction = predict_list[i](models[i], row)
		stacked_row.append(prediction)
	stacked_row.append(row[-1])
	return stacked_row

En algunos problemas de modelado predictivo, es posible obtener un impulso aún mayor entrenando el modelo agregado tanto en la fila de entrenamiento como en las predicciones realizadas por los submodelos. Esta mejora le brinda al modelo agregador el contexto de todos los datos en la fila de entrenamiento para ayudar a determinar cómo y cuándo combinar mejor las predicciones de los submodelos.

Podemos actualizar nuestra función `to_stacked_row()` para incluir esto agregando la fila de entrenamiento (menos la
columna final) y la fila apilada como se creó anteriormente. 


In [6]:
# Make predictions with sub-models and construct a new stacked row
def to_stacked_row(models, predict_list, row):
	stacked_row = list()
	for i in range(len(models)):
		prediction = predict_list[i](models[i], row)
		stacked_row.append(prediction)
	stacked_row.append(row[-1])
	return row[0:len(row)-1] + stacked_row

Es una buena idea probar ambos enfoques en su problema para ver cuál funciona mejor. Ahora que tenemos todas las piezas para la generalización apilada, podemos aplicar el algoritmo a un problema del mundo real.

---
<div style="text-align: right">
<a href="#indice"><font size=5><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></font></a>
</div>

---

<a id="section4"></a> 
# <font color="#004D7F"> 4. Caso de estudio: dataset Sonar</font>

En esta sección, aplicaremos el algoritmo Bagging al conjunto de datos de Sonar. 
1. Primero se carga el conjunto de datos, los valores de cadena se convierten a numéricos y la columna de salida se convierte de cadenas a valores enteros de 0 a 1. 
    - Esto se logra con las funciones auxiliares `load_csv()`, `str_column_to_float()` y `str_columna_to_int()` para cargar y preparar el conjunto de datos.
2. Usaremos _k_-fold para estimar el rendimiento del modelo aprendido en datos invisibles. 
    - Esto significa que construiremos y evaluaremos _k_ modelos y estimaremos el rendimiento como el error medio del modelo. 
    - El Accuracy se utilizará para evaluar cada modelo. 
    - Estos comportamientos se proporcionan en las funciones `cross_validation_split()`, `accuracy_metric()`
y `evaluate_algorithm()`.
3. Usaremos los algoritmos KNN, Perceptrón y regresión logística implementados anteriormente. También usaremos nuestra técnica para crear el nuevo conjunto de datos apilados definido en el paso anterior. Se desarrolla un nuevo nombre de función `stacking()`. Esta función hace 4 cosas:
    1. Primero entrena una lista de modelos (KNN y Perceptron).
    2. Luego utiliza los modelos para hacer predicciones y crear un nuevo conjunto de datos apilados.
    3. Luego entrena un modelo agregador (regresión logística) en el conjunto de datos apilados.
    4. Luego utiliza los submodelos y el modelo agregador para hacer predicciones en el conjunto de datos de prueba. 

El ejemplo completo se enumera a continuación.

In [9]:
# Stacking on the sonar dataset
from random import seed
from random import randrange
from csv import reader
from math import sqrt
from math import exp

# Load a CSV file
def load_csv(filename):
	dataset = list()
	with open(filename, 'r') as file:
		csv_reader = reader(file)
		for row in csv_reader:
			if not row:
				continue
			dataset.append(row)
	return dataset

# Convert string column to float
def str_column_to_float(dataset, column):
	for row in dataset:
		row[column] = float(row[column].strip())

# Convert string column to integer
def str_column_to_int(dataset, column):
	class_values = [row[column] for row in dataset]
	unique = set(class_values)
	lookup = dict()
	for i, value in enumerate(unique):
		lookup[value] = i
	for row in dataset:
		row[column] = lookup[row[column]]
	return lookup

# Split a dataset into k folds
def cross_validation_split(dataset, n_folds):
	dataset_split = list()
	dataset_copy = list(dataset)
	fold_size = int(len(dataset) / n_folds)
	for _ in range(n_folds):
		fold = list()
		while len(fold) < fold_size:
			index = randrange(len(dataset_copy))
			fold.append(dataset_copy.pop(index))
		dataset_split.append(fold)
	return dataset_split

# Calculate accuracy percentage
def accuracy_metric(actual, predicted):
	correct = 0
	for i in range(len(actual)):
		if actual[i] == predicted[i]:
			correct += 1
	return correct / float(len(actual)) * 100.0

# Evaluate an algorithm using a cross validation split
def evaluate_algorithm(dataset, algorithm, n_folds, *args):
	folds = cross_validation_split(dataset, n_folds)
	scores = list()
	for fold in folds:
		train_set = list(folds)
		train_set.remove(fold)
		train_set = sum(train_set, [])
		test_set = list()
		for row in fold:
			row_copy = list(row)
			test_set.append(row_copy)
			row_copy[-1] = None
		predicted = algorithm(train_set, test_set, *args)
		actual = [row[-1] for row in fold]
		accuracy = accuracy_metric(actual, predicted)
		scores.append(accuracy)
	return scores

# Stacked Generalization Algorithm
def stacking(train, test):
	model_list = [knn_model, perceptron_model]
	predict_list = [knn_predict, perceptron_predict]
	models = list()
	for i in range(len(model_list)):
		model = model_list[i](train)
		models.append(model)
	stacked_dataset = list()
	for row in train:
		stacked_row = to_stacked_row(models, predict_list, row)
		stacked_dataset.append(stacked_row)
	stacked_model = logistic_regression_model(stacked_dataset)
	predictions = list()
	for row in test:
		stacked_row = to_stacked_row(models, predict_list, row)
		stacked_dataset.append(stacked_row)
		prediction = logistic_regression_predict(stacked_model, stacked_row)
		prediction = round(prediction)
		predictions.append(prediction)
	return predictions

# Test stacking on the sonar dataset
seed(1)
# load and prepare data
filename = 'data/sonar.all-data.csv'
dataset = load_csv(filename)
# convert string attributes to integers
for i in range(len(dataset[0])-1):
	str_column_to_float(dataset, i)
# convert class column to integers
str_column_to_int(dataset, len(dataset[0])-1)
n_folds = 3
scores = evaluate_algorithm(dataset, stacking, n_folds)
print('Scores: %s' % scores)
print('Mean Accuracy: %.3f%%' % (sum(scores)/float(len(scores))))

Scores: [78.26086956521739, 76.81159420289855, 69.56521739130434]
Mean Accuracy: 74.879%


- Se utilizó un valor _k_ de 5 para la validación cruzada, dando a cada pliegue $\frac{208}{3} = 69.3$ o justo 70 registros que se evaluarán en cada iteración.
- Al ejecutar el ejemplo se imprimen las puntuaciones y la media de las puntuaciones para la configuración final.

---
<div style="text-align: right">
<a href="#indice"><font size=5><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></font></a>
</div>

---

<a id="sectionEj"></a>
<h3><font color="#004D7F" size=6> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#113D68"></i> Ejercicios</font></h3>

Se proponen las siguientes actividades para consolidar el aprendizaje.

# <font color="#004D7F" size=5>Ejercicio 1</font>
__Sintonizar algoritmos__. Los algoritmos utilizados para los submodelos y el modelo agregado en este tutorial no se ajustaron. Explore configuraciones alternativas y vea si puede mejorar aún más el rendimiento.

# <font color="#004D7F" size=5>Ejercicio 2</font>
__Correlaciones de predicción__. El apilamiento funciona mejor si las predicciones de los submodelos están débilmente correlacionadas. Implementar cálculos para estimar la correlación entre las predicciones de los submodelos.

# <font color="#004D7F" size=5>Ejercicio 3</font>
__Diferentes Submodelos__. Implementar más y diferentes submodelos para combinarlos usando el procedimiento de apilamiento.

# <font color="#004D7F" size=5>Ejercicio 4</font>
__Modelo Agregador Diferente__. Experimente con modelos más simples (como promediar y votar) y modelos de agregación más complejos para ver si puede mejorar el rendimiento.

# <font color="#004D7F" size=5>Ejercicio 5</font>
__Más conjuntos de datos__. Aplique el apilamiento a más conjuntos de datos en el [repositorio de aprendizaje automático de UCI](https://archive.ics.uci.edu/).

---

<div style="text-align: right">
<a href="#indice"><font size=5><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></font></a>
</div>

---

<div style="text-align: right"> <font size=6><i class="fa fa-coffee" aria-hidden="true" style="color:#004D7F"></i> </font></div>